# شطر — التقسيم القديم المعتمد
هذا هو الدفتر الافتراضي بناءً على اختيارك: نعيد استخدام بيانات basma_project الأصلية ونموذج AraPoemBERT ذي Accuracy=61.20% وMacro-F1=61.53%. لا نعيد التقسيم أو تدريب المصنف. نتيجة التحقق محفوظة من التجربة الأصلية، لا تقييم جديد.

التقسيم الأصلي يتضمن شاعرين عثمانيين في التحقق وشاعرًا عثمانيًا واحدًا في الاختبار، لذلك لا يحقق شرط الحد الأدنى ثلاثة شعراء الذي جربناه في التقسيم الجديد. لا نغيره لرفع النتيجة. الدفتر الاختياري Shatr_NewSplit_Optional.ipynb منفصل.

In [ ]:
from pathlib import Path
import os,sys,zipfile,subprocess,shutil,json
from google.colab import drive,files
drive.mount('/content/drive')
PACKAGE=Path('/content/Shatr_Project')
if not PACKAGE.exists():
    uploaded=files.upload()
    archive=next((n for n in uploaded if n.lower().endswith('.zip')),None)
    if archive is None:raise ValueError('ارفعي Shatr_Project.zip')
    with zipfile.ZipFile(archive) as z:
        for name in z.namelist():
            if not (Path('/content')/name).resolve().is_relative_to(Path('/content')):raise ValueError('Unsafe archive path')
        z.extractall('/content')
ROOT=Path('/content/shatr_project_legacy')
OLD_ROOT=Path('/content/drive/MyDrive/basma_project')
PERSIST=Path('/content/drive/MyDrive/shatr_project_legacy')
os.environ['SHATR_ROOT']=str(ROOT)
def run(*args):subprocess.run([sys.executable,*args],cwd=PACKAGE,env=os.environ.copy(),check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(PACKAGE/'requirements.txt')],check=True)


## استعادة البيانات والأوزان الأصلية
التحقق يعتمد على تواقيع ملفات التدريب والتحقق والتسميات داخل experiment.json الأصلي. لا ننشئ سجل تدريب مزيفًا للنموذج القديم.

In [ ]:
if not (ROOT/'data/labels.json').exists():
    shutil.copytree(OLD_ROOT/'data',ROOT/'data')
model=ROOT/'outputs/arapoembert_model'
if not (model/'model.safetensors').exists():
    # داخل جلسة العمل الأصلية يمكن استخدام الأوزان المحفوظة بالفعل.
    original_model=Path('/content/basma_arapoembert/model')
    if original_model.exists():
        run('use_old_split.py','--old-root',str(OLD_ROOT),'--model',str(original_model),'--target',str(ROOT))
    else:
        print('ارفعي أرشيف النموذج الأصلي Basma_AraPoemBERT_Model.zip ذي نتيجة 61.20%.')
        uploaded=files.upload()
        filename=next((n for n in uploaded if n.lower().endswith('.zip')),None)
        if filename is None:raise ValueError('ارفعي أرشيف النموذج الأصلي')
        run('restore_model.py','--archive',str(Path(filename).resolve()),'--root',str(ROOT),'--model','arapoembert')
        run('use_old_split.py','--old-root',str(OLD_ROOT),'--model',str(model),'--target',str(ROOT))
else:
    run('use_old_split.py','--old-root',str(OLD_ROOT),'--model',str(model),'--target',str(ROOT))
selection=json.loads((ROOT/'outputs/selected_classifier.json').read_text())
print('النموذج المعتمد:',selection['model_id'])
print('دقة التحقق الأصلية:',format(selection['accuracy'],'.2%'))
print('Macro-F1:',format(selection['macro_f1'],'.2%'))


In [ ]:
run('test_project.py')
run('test_features.py')
run('make_report.py')
run('sync_results.py','--root',str(ROOT),'--dest',str(PERSIST),'--skip-data')


## تدريب التكملة — اختياري ويحتاج GPU
التدريب الكامل والتقييم البشري ما زالا مطلوبين. هذا المسار يستخدم الأزواج القديمة نفسها، ولا يغير بيانات المصنف. اتركي RUN_GENERATION=False إذا أردت تشغيل البصمة فقط.

In [ ]:
RUN_GENERATION=False
if RUN_GENERATION:
    run('train_completion.py','before')
    run('train_completion.py','train','--epochs','2')
    run('train_completion.py','compare')
    run('sync_results.py','--root',str(ROOT),'--dest',str(PERSIST),'--skip-data')


## الواجهة
التصنيف يستخدم المصنف القديم بعد فحص بياناته، والبحث من التدريب القديم فقط. التكملة تبقى قيد التجهيز إلى أن يتوفر adapter مدرب على هذه البيانات.

In [ ]:
sys.path.insert(0,str(PACKAGE))
from app import build_app
from google.colab import output
if 'demo' in globals():demo.close()
demo=build_app(ROOT)
demo.launch(share=False,server_name='0.0.0.0',server_port=7862,prevent_thread_lock=True)
print(output.eval_js('google.colab.kernel.proxyPort(7862)'))


## حفظ الأوزان
احتفظي بأرشيف النموذج الأصلي قبل إنهاء الجلسة. التقارير المحفوظة في Drive لا تشمل ملف أوزان المصنف الكبير. بيانات المصدر نفسها موجودة أصلًا في MyDrive/basma_project/data فلا نكررها في Drive. لا نخلط تقارير أو نتائج التقسيم الجديد مع هذا النموذج.